# Hands-on: Comparação de Variantes do SARS-CoV-2

---

## Contexto biológico

Em 2021, surgiram várias variantes do SARS-CoV-2 com mutações que alteraram a transmissibilidade e a capacidade de escape imune do vírus. Uma das mais estudadas foi a variante **Delta (B.1.617.2)**, identificada pela primeira vez na Índia.

Neste trabalho vais comparar dois genomas completos:

| Ficheiro | Accession | Descrição |
|---|---|---|
| `SARS-CoV-2_ref_aln.fasta` | NC_045512.2 | Wuhan-Hu-1 (estirpe de referência, 2019) |
| `SARS-CoV-2_var_aln.fasta` | MZ359844.1  | Variante indiana, 2021 |

> **Nota:** Os dois ficheiros já estão **pré-alinhados** com o algoritmo de Needleman-Wunsch (alinhamento global). Isso significa que as duas sequências têm exatamente o mesmo comprimento e que os `-` representam gaps — posições onde uma sequência tem uma inserção/deleção relativamente à outra.

---

## O que vais construir

```
FASTA alinhados
      ↓
 Exercício 1 — Ler as sequências
      ↓
 Exercício 2 — Explorar o alinhamento (matches, mismatches, gaps)
      ↓
 Exercício 3 — Divergência por janelas de 1 000 bp
      ↓
 Exercício 4 — Visualizar os resultados
      ↓
 Exercício 5 — Interpretação biológica
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

---
## Exercício 1 — Ler ficheiros FASTA

O formato FASTA é o mais comum para armazenar sequências biológicas. Cada sequência começa com uma linha de cabeçalho que começa por `>`, seguida das linhas com a sequência em si.

```
>NC_045512.2 Severe acute respiratory syndrome coronavirus 2 ...
ATTAAAGGTTTATACCTTCCCAGGTAACAAACCAACCAACTTTCGATCT...
CGAACTTTAAAATCTGTGTGGCTGTCACTCGGCTGCATGCTTAGTGCAC...
```

**Tarefa:** Implementa a função `read_fasta` que:
1. Abre o ficheiro no caminho `path`
2. Guarda o cabeçalho (linha que começa por `>`, sem o `>`)
3. Concatena todas as linhas de sequência em maiúsculas numa única string
4. Devolve `(header, sequence)`

In [ ]:
def read_fasta(path):
    """
    Lê um ficheiro FASTA e devolve o cabeçalho e a sequência.

    Parâmetros
    ----------
    path : str
        Caminho para o ficheiro .fasta

    Devolve
    -------
    header : str
    sequence : str  (apenas caracteres de nucleótido + '-' para gaps)
    """
    # TODO: implementa esta função
    pass

In [ ]:
# ── Verificação ───────────────────────────────────────────────────────────────
REF_PATH = "../SARS-CoV-2_ref_aln.fasta"
VAR_PATH = "../SARS-CoV-2_var_aln.fasta"

hdr_ref, seq_ref = read_fasta(REF_PATH)
hdr_var, seq_var = read_fasta(VAR_PATH)

print(f"Referência : {hdr_ref[:70]}")
print(f"Comprimento: {len(seq_ref):,} colunas")
print()
print(f"Variante   : {hdr_var[:70]}")
print(f"Comprimento: {len(seq_var):,} colunas")
print()

# As duas sequências devem ter o mesmo comprimento (estão alinhadas)
assert len(seq_ref) == len(seq_var), "ERRO: comprimentos diferentes — revê a função read_fasta!"
print("✓ Comprimentos iguais — alinhamento carregado corretamente.")

---
## Exercício 2 — Explorar o alinhamento

Com as duas sequências alinhadas (mesmo comprimento), cada posição `i` forma uma **coluna de alinhamento**. Cada coluna pode ser:

| Tipo | Exemplo | Significado |
|---|---|---|
| **Match** | `A` / `A` | Base conservada entre as duas estirpes |
| **Mismatch** | `A` / `T` | Substituição (SNP) |
| **Gap** | `A` / `-` ou `-` / `A` | Inserção ou deleção (indel) |

**Tarefa:** Preenche a função `explore_alignment` para contar o número de matches, mismatches e gaps no alinhamento completo, e calcula a **identidade global** (% de matches).

In [ ]:
def explore_alignment(seq1, seq2):
    """
    Conta matches, mismatches e gaps entre duas sequências alinhadas.

    Devolve
    -------
    dict com chaves: 'matches', 'mismatches', 'gaps', 'identity_pct'
    """
    matches, mismatches, gaps = 0, 0, 0

    for r, v in zip(seq1, seq2):
        if r == '-' or v == '-':
            # TODO: incrementa gaps
            pass
        elif r == v:
            # TODO: incrementa matches
            pass
        else:
            # TODO: incrementa mismatches
            pass

    total = len(seq1)
    # TODO: calcula identidade (matches / total * 100)
    identity_pct = 0

    return {
        "matches":      matches,
        "mismatches":   mismatches,
        "gaps":         gaps,
        "identity_pct": identity_pct,
    }

In [ ]:
stats = explore_alignment(seq_ref, seq_var)

print("=== Estatísticas globais do alinhamento ===")
print(f"  Comprimento total : {len(seq_ref):,} colunas")
print(f"  Matches           : {stats['matches']:,}")
print(f"  Mismatches (SNPs) : {stats['mismatches']:,}")
print(f"  Gaps (indels)     : {stats['gaps']:,}")
print(f"  Identidade global : {stats['identity_pct']:.2f} %")
print()
print("Preview do alinhamento (posições 100–180):")
ref_preview = seq_ref[100:180]
var_preview = seq_var[100:180]
bar = "".join("|" if r == v and r != '-' else " " for r, v in zip(ref_preview, var_preview))
print(f"  REF: {ref_preview}")
print(f"       {bar}")
print(f"  VAR: {var_preview}")

---
## Exercício 3 — Divergência por janelas deslizantes

A identidade global diz-nos **quanto** as duas sequências diferem, mas não **onde**. Para encontrar as regiões mais mutadas, usamos uma **janela deslizante**: percorremos o alinhamento em blocos de tamanho fixo (aqui, 1 000 colunas) e contamos, em cada bloco, quantas colunas são divergentes (mismatch ou gap).

```
Alinhamento completo (~30 000 colunas)
│◄────────── janela 1 (1000) ──────────►│◄────── janela 2 ──────►│ ...
```

Para cada janela precisamos também de saber a **posição no genoma de referência** — não na contagem de colunas do alinhamento, pois os gaps da referência não correspondem a posições reais.

**Tarefa:** Completa a função `windowed_divergence`.

In [ ]:
def windowed_divergence(seq_ref_aln, seq_var_aln, window=1000):
    """
    Calcula a divergência em janelas não sobrepostas de `window` colunas.

    Parâmetros
    ----------
    seq_ref_aln, seq_var_aln : str
        Sequências alinhadas (mesmo comprimento, com '-' nos gaps)
    window : int
        Tamanho da janela em colunas de alinhamento

    Devolve
    -------
    positions_ref : list[int]
        Posição central de cada janela em coordenadas da referência
    divergence : list[int]
        Número de colunas divergentes (mismatch ou gap) por janela
    """
    aln_len = len(seq_ref_aln)

    # Constrói o mapa: índice de coluna → posição na referência
    # (as colunas onde seq_ref_aln == '-' não contam como posição real)
    ref_coord = 0
    ref_pos_map = []
    for c in seq_ref_aln:
        ref_pos_map.append(ref_coord)
        if c != '-':
            ref_coord += 1

    positions_ref, divergence = [], []

    for aln_start in range(0, aln_len - window + 1, window):
        aln_end = aln_start + window

        # TODO: extrai as janelas das duas sequências alinhadas
        chunk_ref = ...
        chunk_var = ...

        # TODO: conta colunas divergentes (quando r != v)
        div = ...

        # Posição central da janela em coordenadas da referência
        ref_mid = ref_pos_map[aln_start + window // 2]

        positions_ref.append(ref_mid)
        divergence.append(div)

    return positions_ref, divergence

In [ ]:
WINDOW = 1000

positions_ref, divergence = windowed_divergence(seq_ref, seq_var, window=WINDOW)

divergence_per_bp = [d / WINDOW for d in divergence]
cumulative        = list(np.cumsum(divergence_per_bp))

print(f"Janelas analisadas  : {len(positions_ref)}")
print(f"Divergência total   : {sum(divergence):,} colunas")
print()
print("Top 5 janelas mais divergentes:")
ranked = sorted(zip(divergence_per_bp, positions_ref), reverse=True)
for dpb, pos in ranked[:5]:
    print(f"  ~{pos:>6,} nt na referência  →  {dpb:.4f} div/bp  ({int(dpb*WINDOW)} colunas divergentes)")

---
## Exercício 4 — Visualização

**Tarefa A (base):** Cria um gráfico de barras com a divergência por janela (`divergence_per_bp`) no eixo Y e a posição na referência (`positions_ref`) no eixo X.

**Tarefa B (desafio):** Assinala a região **Spike** (S) a vermelho. As coordenadas no genoma de referência NC_045512.2 são:
- Spike: posições **21 563 – 25 384 nt**

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# TODO Tarefa A: cria um gráfico de barras
# Dica: ax.bar(x, height, width=...)


# TODO Tarefa B (desafio): pinta de vermelho as barras dentro da Spike
# Dica: cria uma lista de cores com uma list comprehension
# e passa-a ao parâmetro `color` do ax.bar
SPIKE_START, SPIKE_END = 21_563, 25_384


ax.set_xlabel("Posição no genoma de referência (nt)", fontsize=11)
ax.set_ylabel("Divergência por bp", fontsize=11)
ax.set_title("Divergência por janelas de 1 000 bp — SARS-CoV-2 Wuhan vs. Variante Indiana")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

---
## Exercício 5 — Gráfico completo com contexto biológico

Abaixo tens o código para o gráfico final com 3 painéis:
1. **Divergência cumulativa** — mostra como a distância ao genoma de referência acumula ao longo do genoma
2. **Divergência por janela** — com os subdomínios da Spike assinalados
3. **Mapa de genes** — todos os ORFs do SARS-CoV-2 nas suas posições reais

Analisa o código, corre-o, e responde às perguntas em seguida.

In [ ]:
# ── Anotações biológicas (coordenadas NC_045512.2) ────────────────────────────
GENES = [
    ("ORF1ab", 266,   21555, "#4e79a7"),
    ("S",      21563, 25384, "#e15759"),
    ("ORF3a",  25393, 26220, "#f28e2b"),
    ("E",      26245, 26472, "#76b7b2"),
    ("M",      26523, 27191, "#59a14f"),
    ("ORF6",   27202, 27387, "#edc948"),
    ("ORF7a",  27394, 27759, "#b07aa1"),
    ("ORF7b",  27756, 27887, "#ff9da7"),
    ("ORF8",   27894, 28259, "#9c755f"),
    ("N",      28274, 29533, "#bab0ac"),
    ("ORF10",  29558, 29674, "#d4a6c8"),
]

SPIKE_DOMAINS = [
    ("NTD", 21563, 22599, "#ffb3b3"),
    ("RBD", 22517, 23183, "#ff4444"),
    ("S2",  23623, 25384, "#ff8080"),
]
FURINA_POS = 23603

# ── Figura ────────────────────────────────────────────────────────────────────
fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1, figsize=(16, 11), sharex=True,
    gridspec_kw={"height_ratios": [1.8, 1.6, 0.55]}
)
fig.suptitle(
    "Divergência SARS-CoV-2: Wuhan-Hu-1 vs. Variante Indiana MZ359844.1\n"
    "Alinhamento Needleman-Wunsch · Janelas de 1 000 bp · Coordenadas NC_045512.2",
    fontsize=13, fontweight="bold"
)

x          = positions_ref
genome_len = 29_903

# Painel 1 — divergência cumulativa
ax1.plot(x, cumulative, color="#1f77b4", linewidth=2,
         label="Divergência cumulativa (por bp)")
ax1.axvspan(21563, 25384, alpha=0.12, color="#e15759", zorder=0, label="Região Spike")
ax1.set_ylabel("Divergência\ncumulativa / bp", fontsize=10)
ax1.legend(fontsize=9, loc="upper left")
ax1.grid(axis="y", linestyle="--", alpha=0.35)
max_idx = int(np.argmax(divergence_per_bp))
ax1.annotate(
    f"Pico: ~{x[max_idx]:,} nt\n({divergence_per_bp[max_idx]:.3f} div/bp)",
    xy=(x[max_idx], cumulative[max_idx]),
    xytext=(x[max_idx] - 5000, cumulative[max_idx] * 0.80),
    arrowprops=dict(arrowstyle="->", color="#333"),
    fontsize=8.5, bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8)
)

# Painel 2 — barras por janela + subdomínios da Spike
for name, s, e, color in SPIKE_DOMAINS:
    ax2.axvspan(s, e, alpha=0.22, color=color, zorder=0)
    ax2.text((s + e) / 2, max(divergence_per_bp) * 1.03, name,
             ha="center", va="bottom", fontsize=7.5,
             color="#6b0000", fontweight="bold")
ax2.axvline(FURINA_POS, color="#8b0000", linewidth=1.2, linestyle="--", zorder=3)
ax2.text(FURINA_POS + 80, max(divergence_per_bp) * 0.97,
         "Furina\nP681R", fontsize=6.5, color="#8b0000", va="top")
bar_colors = ["#e15759" if 21563 <= p <= 25384 else "#aec7e8" for p in x]
ax2.bar(x, divergence_per_bp, width=WINDOW * 0.85,
        color=bar_colors, align="center", zorder=2)
ax2.set_ylabel("Divergência\npor bp / janela", fontsize=10)
ax2.grid(axis="y", linestyle="--", alpha=0.35)
ax2.legend(handles=[
    mpatches.Patch(color="#e15759", alpha=0.8, label="Spike (S)"),
    mpatches.Patch(color="#aec7e8",            label="Restante genoma"),
    plt.Line2D([0],[0], color="#8b0000", linestyle="--", label="Sítio furina (~23 603 nt)"),
], fontsize=8.5, loc="upper left")

# Painel 3 — mapa de genes
ax3.set_ylim(0, 1)
ax3.set_yticks([])
ax3.set_ylabel("Genes", fontsize=9)
ax3.set_xlabel("Posição no genoma de referência (nt)", fontsize=11)
for name, s, e, color in GENES:
    ax3.barh(0.5, e - s, left=s, height=0.55, color=color,
             align="center", edgecolor="white", linewidth=0.4)
    if e - s > 600:
        ax3.text((s + e) / 2, 0.5, name,
                 ha="center", va="center", fontsize=7,
                 color="white", fontweight="bold")
ax3.set_xlim(0, genome_len)
ax3.grid(False)

plt.tight_layout(h_pad=0.6)
plt.savefig("../grafico_spike.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Exercício 6 — Interpretação biológica

Responde às seguintes questões com base nos resultados obtidos:

**Q1.** Qual é a identidade global entre os dois genomas? O que significa esse valor em termos evolutivos?

**Q2.** Quantos gaps existem no alinhamento? Em qual das duas sequências se encontram maioritariamente? O que representa biologicamente um gap num alinhamento de genomas virais?

**Q3.** Olhando para o gráfico de barras (Exercício 4 / Painel 2 do Exercício 5), qual a região do genoma com maior divergência? Coincide com alguma região biologicamente relevante?

**Q4.** A proteína Spike tem três subdomínios representados no gráfico: **NTD**, **RBD** e **S2**. Com base nas barras dentro da Spike, qual dos subdomínios parece ser o mais divergente? Qual a relevância desse subdomínio para a resposta imune?

**Q5.** O sítio de clivagem por furina (linha tracejada a ~23 603 nt) está associado à mutação **P681R** na variante Delta. Porquê é que uma mutação neste sítio pode aumentar a infectividade do vírus?

**Q6. (desafio)** O ORF1ab é o gene mais longo do genoma (~21 kb) mas apresenta baixa divergência relativa. Como se pode explicar isso biologicamente?

### As tuas respostas

**Q1.**

**Q2.**

**Q3.**

**Q4.**

**Q5.**

**Q6.**